# VoltaSegura — pergunta de ML com dados reais (SSP-SP + Infosiga)

**Disciplina**: Inteligência Artificial & Machine Learning (GS FIAP — tema CitizenTech)
**Produto**: VoltaSegura (ex-"CitizenTech", renomeado em 10/11/2026)

## Pergunta de ML

> **Dado um dia da semana, uma faixa horária e um bairro do município de São Paulo,
> qual é a frequência esperada (relativa) de BOs de roubo/furto — tendo os
> sinistros de trânsito do Infosiga como contraprova multimodal — e em quais
> células da grade os dados são insuficientes para qualquer conclusão?**

## Guardrails de comunicação (docs/05 — OBRIGATÓRIOS)

- NUNCA rotular bairro/distrito como "seguro" ou "perigoso".
- NUNCA prometer "rota segura" — o produto comunica **exposição estimada** e **confiança dos dados**.
- Sempre comparações/valores relativos acompanhados do volume de dados que sustenta a estimativa.
- Zero registros ≠ seguro (**subnotificação**) — tratado explicitamente na seção de limitações.

## Fontes (reais, públicas, documentadas em `dados/README.md`)

| Fonte | Recurso | Período | Licença |
|---|---|---|---|
| SSP-SP "SPDadosCriminais" — BOs individuais (bairro, lat/lon, rubrica, hora) | https://www.ssp.sp.gov.br/estatistica/consultas | 2026 (abas JAN-JUN e JUL-DEZ) | CC-BY-4.0 (via CKAN estadual) |
| Infosiga SP — sinistros de trânsito (logradouro, lat/lon, hora, modo) | https://infosiga.detran.sp.gov.br/ → Dados Abertos | 2015–2026 | CC-BY-4.0 |

## Mapa do notebook → rubrica da disciplina

| Critério da rubrica | Seção |
|---|---|
| Problema e dados reais (2,0) | Pergunta (capa) + §1 |
| Preparação e divisão treino/teste (2,0) | §2 |
| Escolha justificada do modelo (1,0) | §3 |
| Treino com diferentes parâmetros (2,0) | §4 — 27 combinações de 3 hiperparâmetros, CV **só no treino** |
| Resultados em treino e teste (2,0) | §5 — métricas lado a lado + gráficos + overfitting |
| Conclusão e leitura para o cidadão (1,0) | §6–§8 |

## Como executar

```bash
# na raiz do repo (arquivos brutos conforme src/ml/dados/README.md — SHAs lá):
uv run --with pandas,openpyxl,pyarrow,scikit-learn,matplotlib,nbclient,nbformat,ipykernel   jupyter execute src/ml/notebook_ml.ipynb
# ou no VS Code/Jupyter Lab normalmente (Python ≥ 3.11).
```

O `etl.py` (neste diretório) carrega/consolida as fontes reais com cache de parquet
(`src/ml/dados/cache/`, fora do git — regenerável).

In [1]:
import sys, warnings, gc
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.sparse import csr_matrix

import etl  # módulo local: carrega/consolida as fontes reais (cache em dados/cache/)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10
pd.set_option("display.width", 170)

IMAGENS = Path("imagens")
IMAGENS.mkdir(exist_ok=True)

print("Python", sys.version.split()[0])
print("pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__)
print("raiz de dados detectada:", etl.data_root())

Python 3.11.18
pandas 3.1.1 | numpy 2.4.6 | scikit-learn 1.9.1 | matplotlib 3.11.2
raiz de dados detectada: /home/asteriuz/projects/VoltaSegura


## 1. Carga dos dados reais

Se o cache de parquet não existir, `etl.build_cache()` reconstrói a partir dos
arquivos brutos (baixáveis conforme `dados/README.md`, com SHA256 dos originais
registrados lá). O Infosiga é filtrado para o município de São Paulo
(IBGE 3550308) já na carga; a base da SSP-SP cobre todo o estado e será
recortada na seção 2.

In [2]:
caches = etl.build_cache(force=False)

ssp = pd.read_parquet(caches["ssp_bos_2026"]) if "ssp_bos_2026" in caches else None
sinistros = pd.read_parquet(caches["infosiga_sinistros_sp"]) if "infosiga_sinistros_sp" in caches else None

def resumo(nome, df):
    if df is None:
        print(f"[avoitado] {nome}: base bruta não encontrada — seção correspondente é omitida")
        return None
    print(f"[ok] {nome}: {len(df):,} registros | {len(df.columns)} colunas")
    return df

ssp = resumo("SSP-SP BOs 2026 (estado)", ssp)
sinistros = resumo("Infosiga capital 2015-2026", sinistros)

[cache] infosiga_sinistros_sp.parquet já existe — use force=True (abaixo) para regerar
[cache] ssp_bos_2026.parquet já existe — use force=True (abaixo) para regerar


[ok] SSP-SP BOs 2026 (estado): 814,900 registros | 13 colunas
[ok] Infosiga capital 2015-2026: 343,241 registros | 23 colunas


## 2. Preparação (SSP-SP = base principal)

Recortes documentados e reprodutíveis:

1. **Município**: apenas a capital (`NOME_MUNICIPIO == "S.PAULO"`).
2. **Natureza do evento**: `RUBRICA` contém `ROUBO` ou `FURTO` — crimes de
   agravador–vítima com vítima presente, insumo declarado de exposição
   (o produto nunca os usa como rótulo territorial).
3. **Grade**: `bairro × dia da semana × faixa horária × mês`, com zeros
   explícitos nas células sem evento — para o modelo distinguir "frequente"
   de "sem registro".
4. **Split temporal** (justificação): treino = meses 1–4/2026; teste = meses
   5–6/2026. O teste nunca entra na validação cruzada; a divisão por tempo
   simula a tarefa real: estimar o mês seguinte sem conhecer o futuro.

In [3]:


def hora_para_int(v):
    # HORA_OCORRENCIA_BO chega como texto 'HH:MM[:SS]' (ou nulo)
    s = str(v).strip()
    if s in ("", "NaN", "NaT", "None", "nan"):
        return np.nan
    for fmt in ("%H:%M:%S", "%H:%M"):
        try:
            return pd.to_datetime(s, format=fmt).hour
        except Exception:
            continue
    return np.nan

if ssp is None:
    print("[seção omitida — base SSP não disponível]")
else:
    capital = ssp[ssp["NOME_MUNICIPIO"].str.strip().str.upper().eq("S.PAULO")].copy()
    rubrica = capital["RUBRICA"].astype(str).str.upper()
    crimes = capital[rubrica.str.contains("ROUBO") | rubrica.str.contains("FURTO")].copy()

    crimes["hora"] = crimes["HORA_OCORRENCIA_BO"].map(hora_para_int)
    crimes["faixa"] = crimes["hora"].map(etl.faixa_horaria)
    crimes["dia"] = crimes["DATA_OCORRENCIA_BO"].dt.dayofweek   # 0=segunda … 6=domingo
    crimes["mes"] = pd.to_numeric(crimes["MES_ESTATISTICA"], errors="coerce")
    crimes["BAIRRO"] = crimes["BAIRRO"].fillna("(sem bairro informado)").astype(str).str.strip().str.upper()

    n_sem_bairro = (crimes["BAIRRO"] == "(SEM BAIRRO INFORMADO)").sum()
    print(f"BOs na capital: {len(capital):,}")
    print(f"BOs de roubo/furto: {len(crimes):,} ({len(crimes)/len(capital):.1%} da capital)")
    print("Período:", crimes['DATA_OCORRENCIA_BO'].min().date(), "→", crimes['DATA_OCORRENCIA_BO'].max().date())
    print(f"  sem bairro informado   : {n_sem_bairro:,}")
    print(f"  sem hora utilizável    : {crimes['hora'].isna().sum():,}")

    DIAS = list(range(7))
    FAIXAS = list(etl.faixa_horaria(h) for h in (0, 6, 12, 18))
    # meses 1-6 completos; 7-9/2026 chegam parciais na base (aba JUL-DEZ em
    # atualização) — excluídos INTEIROS da grade para não enviesar treino/teste.
    MESES = sorted(int(m) for m in crimes["mes"].dropna().unique() if m <= 6)
    BAIRROS = sorted(crimes["BAIRRO"].unique())

    grupo = (
        crimes.groupby(["BAIRRO", "dia", "faixa", "mes"], dropna=True)
        .size().rename("ocorrencias").reset_index()
    )
    grade = pd.MultiIndex.from_product(
        [BAIRROS, DIAS, FAIXAS, MESES], names=["BAIRRO", "dia", "faixa", "mes"],
    ).to_frame(index=False)
    dados = grade.merge(grupo, on=["BAIRRO", "dia", "faixa", "mes"], how="left").fillna({"ocorrencias": 0})
    dados["ocorrencias"] = dados["ocorrencias"].astype(int)

    print(f"\nGrade completa: {len(dados):,} células (bairro × dia × faixa × mês)")
    display(dados["ocorrencias"].describe().to_frame().T.round(2))
    display(dados.head(3))

BOs na capital: 330,690
BOs de roubo/furto: 274,990 (83.2% da capital)
Período: 2006-02-25 → 2026-09-30
  sem bairro informado   : 22
  sem hora utilizável    : 109,367



Grade completa: 375,984 células (bairro × dia × faixa × mês)


,count,mean,std,min,25%,50%,75%,max
ocorrencias,375984.0,0.31,1.55,0.0,0.0,0.0,0.0,124.0


,BAIRRO,dia,faixa,mes,ocorrencias
0,,0,MADRUGADA (00-05),1,0
1,,0,MADRUGADA (00-05),2,0
2,,0,MADRUGADA (00-05),3,2


### Onde os dados são insuficientes (query exigida pela pergunta de ML)

Agrade de análise permite quantificar a confiabilidade por célula: quantos bairros
têm menos de K registros no período?  Isso alimenta a "confiança" exibida no
produto: bairros com poucas observações ficam marcados, em vez de receberem
zero (que seria lido — incorretamente — como "seguro").

In [4]:
if ssp is not None:
    freq_bairro = crimes.groupby("BAIRRO").size()
    for k in (10, 50, 100):
        v = (freq_bairro < k).sum()
        print(f"  bairros com < {k:>3} BOs de roubo/furto em 2026: {v:>4} de {len(freq_bairro):,} "
              f"({v/len(freq_bairro):.0%})")
    print("\nQuartis de registros por bairro:")
    display(freq_bairro.describe(percentiles=[.25, .5, .75]).round(1))

  bairros com <  10 BOs de roubo/furto em 2026: 1051 de 2,238 (47%)
  bairros com <  50 BOs de roubo/furto em 2026: 1756 de 2,238 (78%)
  bairros com < 100 BOs de roubo/furto em 2026: 1947 de 2,238 (87%)

Quartis de registros por bairro:


count    2238.0
mean      122.9
std       493.0
min         1.0
25%         3.0
50%        11.0
75%        39.0
max      8395.0
dtype: float64

## 3. Escolha justificada do modelo — RandomForestRegressor

**Por que regressão de contagens (transformação log1p)?** A pergunta é "quanta
frequência", não "qual classe tex". O produto exibe comparações relativas
(mais/menos eventos estimados), não rótulos.

**Por que Random Forest?**
1. Captura **interações não lineares** (bairro × faixa × dia sem gráfico à priori);
2. Robusto a outliers e à assimetria das contagens (após log1p);
3. Pouco preprocessamento (sem escala/normalização);
4. Fornece **importância de features** — exigência de explicabilidade;
5. Alternativas: **XGBoost** demandaria mais tuning fora do escopo; **MLP/LSTM**
   não superam tabulares nessa tarefa; **KMeans+PCA** seria cluster não
   supervisionado — não responde à pergunta de frequência.

**Grade de hiperparâmetros (3 × 3 × 3 = 27 combinações — rubrica pede ≥10):**

| Hiperparâmetro | Valores | Role |
|---|---|---|
| `n_estimators` | 50, 100, 200 | mais árvores = menos variância, mais custo |
| `max_depth` | 6, 10, None | controla complexidade → overfitting |
| `min_samples_leaf` | 5, 20, 50 | suaviza células pouco representadas |

**Validação cruzada: TimeSeriesSplit(n_splits=3), aplicada SOMENTE ao treino**
(rubrica): os folds são em ordem temporal, o teste (meses 5–6) fica fora e
jamais informa a seleção.

**Detalhe de engenharia (fixado e justificado)**: `max_features="sqrt"`,
`max_samples=30_000` e `n_jobs=1` por árvore. O default de regressão do RF é
`max_features=1.0` — com ~2.2 mil features de bairro, cada nó avaliaria todas
elas e o custo explodiria; com raiz quadrada (~47 features por nó) o custo cai
~50× mantendo a diversidade entre árvores. `max_samples` treina cada árvore
numa subamostra bootstrap de 30 mil células (~8% do treino) — reduz o custo
por árvore ~8× com ganho de diversidade (bagging mais efetivo); a contagem
completa da célula permanece no target, e o teste continua integral. O
paralelismo fica no `GridSearchCV` (27 combinações), não dentro de cada fit.

Função única reutilizada §4 (SSP) e §6 (Infosiga), garantindo o mesmo protocolo
em ambas as fontes; a matriz fica **esparso (CSR)** — com ~2.2 mil colunas de
bairro e zeros explícitos, a versão densa ultrapassaria 5 GB por fold.

In [5]:
GRADE_HIPERPARAMS = {
    "n_estimators": [50, 100, 200],
    "max_depth": [6, 10, None],
    "min_samples_leaf": [5, 20, 50],
}


def treinar_e_avaliar(df, feat_cols, target_col, split_col, treino_val, teste_val, nome_fonte=""):
    # ordena o dataframe pelo tempo (necessário para TimeSeriesSplit)
    g = df.copy().sort_values(split_col).reset_index(drop=True)

    # one-hot nos categorizáveis. Com ~2.2 mil bairros, a matriz densa passaria
    # de 5 GB por fold — usamos CSR esparso (3 não-zeros por linha: bairro, dia,
    # faixa), viável em memória e aceito direto pelo RandomForest.
    X_dense = pd.get_dummies(g[feat_cols], columns=feat_cols)
    feature_names = list(X_dense.columns)
    X = csr_matrix(X_dense.to_numpy(dtype=np.uint8))
    del X_dense
    y = g[target_col].astype(float)
    is_train = g[split_col].isin(list(treino_val))
    X_tr, y_tr = X[is_train.values], y[is_train.values]
    X_te, y_te = X[~is_train.values], y[~is_train.values]

    cv = TimeSeriesSplit(n_splits=3)
    busca = GridSearchCV(
        RandomForestRegressor(
            random_state=42, max_features="sqrt", max_samples=30_000, n_jobs=1,
        ),
        GRADE_HIPERPARAMS,
        cv=cv, scoring="neg_root_mean_squared_error", n_jobs=2, verbose=0,
    )
    busca.fit(X_tr, y_tr)
    gc.collect()

    melhor = busca.best_estimator_
    pred_tr = melhor.predict(X_tr)
    pred_te = melhor.predict(X_te)

    def metricas(yv, pv):
        return {
            "RMSE": float(np.sqrt(mean_squared_error(yv, pv))),
            "MAE": float(mean_absolute_error(yv, pv)),
            "R2": float(r2_score(yv, pv)),
        }

    res = pd.DataFrame({"treino": metricas(y_tr, pred_tr), "teste": metricas(y_te, pred_te)}).T
    print(f"\n[{nome_fonte}] melhores hiperparâmetros: {busca.best_params_}")
    print(f"[{nome_fonte}] células treino/teste: {len(y_tr):,} / {len(y_te):,}")
    display(res.round(4))

    return {
        "busca": busca, "modelo": melhor,
        "pred_treino": pred_tr, "pred_teste": pred_te,
        "y_treino": y_tr, "y_teste": y_te,
        "X_treino": X_tr, "X_teste": X_te,
        "feature_names": feature_names,
        "res": res, "g": g, "feat_cols": feat_cols, "nome": nome_fonte,
    }


r_ssp = r_infosiga = None
if ssp is not None:
    ss = dados.copy()
    ss["target_log"] = np.log1p(ss["ocorrencias"])
    r_ssp = treinar_e_avaliar(
        ss, ["BAIRRO", "dia", "faixa"], "target_log", "mes",
        treino_val=[1, 2, 3, 4], teste_val=[5, 6], nome_fonte="SSP-SP",
    )


[SSP-SP] melhores hiperparâmetros: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}
[SSP-SP] células treino/teste: 250,656 / 125,328


,RMSE,MAE,R2
treino,0.2248,0.1123,0.6909
teste,0.2424,0.1187,0.6129


## 5. Resultados em treino e teste (lado a lado) + leitura de overfitting

Métricas calculadas sobre `log1p(ocorrências)` — evita que células raras
dominem a soma de erros em escala linear.

In [6]:
def figuras_base(r, sufixo):
    nome = r["nome"].lower().replace("-", "_") + f"_{sufixo}"

    y_tr, p_tr = r["y_treino"].values, r["pred_treino"]
    y_te, p_te = r["y_teste"].values, r["pred_teste"]

    # (a) previsto vs real
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    ax.scatter(y_tr, p_tr, s=2, alpha=.12, color="#9aa5b1", label="treino")
    ax.scatter(y_te, p_te, s=4, alpha=.5, color="#f67621", label="teste")
    lo = min(y_tr.min(), y_te.min())
    hi = max(y_tr.max(), y_te.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel("log1p(ocorrências) — real")
    ax.set_ylabel("previsto")
    ax.set_title(f"{r['nome']}: previsto vs real"); ax.legend()
    fig.savefig(IMAGENS / f"pred_vs_real_{nome}.png", bbox_inches="tight")
    plt.close(fig)

    # (b) métricas treino vs teste
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    x = np.arange(3); w = 0.36
    ax.bar(x - w / 2, [r['res'].loc['treino', k] for k in ("RMSE", "MAE", "R2")], w, label="treino", color="#4c78a8")
    ax.bar(x + w / 2, [r['res'].loc['teste', k] for k in ("RMSE", "MAE", "R2")], w, label="teste", color="#f67621")
    ax.set_xticks(x); ax.set_xticklabels(["RMSE", "MAE", "R²"])
    ax.set_title(f"{r['nome']}: métricas treino vs teste"); ax.legend()
    fig.savefig(IMAGENS / f"metricas_{nome}.png", bbox_inches="tight")
    plt.close(fig)

    # (c) importância das features (top 12)
    imp = pd.Series(r["modelo"].feature_importances_, index=r["feature_names"])
    top = imp.sort_values(ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    top.iloc[::-1].plot.barh(ax=ax, color="#4c78a8")
    ax.set_title(f"{r['nome']}: 12 features mais importantes")
    fig.savefig(IMAGENS / f"importancias_{nome}.png", bbox_inches="tight")
    plt.close(fig)
    display(top.to_frame("importância").head(6).round(4))

if r_ssp is not None:
    figuras_base(r_ssp, "rua")

,importância
BAIRRO_SANTO AMARO,0.0282
BAIRRO_PINHEIROS,0.0254
BAIRRO_VILA MARIANA,0.0242
BAIRRO_CAPAO REDONDO,0.0230
BAIRRO_BELA VISTA,0.0223
BAIRRO_IPIRANGA,0.0204


### Como ler o overfitting

- **RMSE(teste) ≲ 1,2 × RMSE(treino)** → generalização saudável;
- RMSE(teste) muito acima de treino (e R² de teste muito menor) → o modelo
  memoriza os padrões específicos dos meses 1–4 do treino; aqui os
  `max_depth` e `min_samples_leaf` menores da grade valem como antídoto
  (por isso 27 combinações, e não uma só);
- R² negativo no teste significaria que a média do treino supera o modelo
  devolvido — sinal para reavaliar recortes e features (nunca aceitável sem
  análise).

## 6. Contraprova multimodal — Infosiga (2015–2026)

O Infosiga traz o **modo de deslocamento envolvido** (`qtd_pedestre`,
`qtd_bicicleta`, `qtd_motocicleta`…) numa série longa. Uso no produto:

1. Contexto por modo e **turno** (madrugada = janela da persona do estudante noturno);
2. Marcação de logradouros com dados insuficientes;
3. Modelo secundário com o **mesmo protocolo** — robustez entre naturezas de evento.

In [7]:
if sinistros is None:
    print("[seção omitida — base Infosiga não disponível]")

if sinistros is not None:
    si = sinistros.copy()
    si["dia"] = si["data_sinistro"].dt.dayofweek
    si["faixa"] = si["hora_sinistro"].map(etl.faixa_horaria)

    modo_cols = ["qtd_pedestre", "qtd_bicicleta", "qtd_motocicleta", "qtd_automovel", "qtd_onibus", "qtd_caminhao"]
    por_ano = si.groupby("ano_sinistro")[modo_cols].sum().astype(int)

    # (d) composição por modo ao longo do tempo
    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    for c in ["qtd_pedestre", "qtd_bicicleta", "qtd_motocicleta", "qtd_automovel"]:
        ax.plot(por_ano.index.astype(int), por_ano[c], marker="o", ms=3, label=c.replace("qtd_", ""))
    ax.set_title("Infosiga capital: envolvidos por modo, 2015–2026")
    ax.set_ylabel("pessoas/veículos envolvidos"); ax.legend(ncol=2, fontsize=8)
    fig.savefig(IMAGENS / "infosiga_modos.png", bbox_inches="tight")
    plt.close(fig)
    display(por_ano.tail(4))

    # turno — MADRUGADA é a janela da persona
    turno_share = (si["turno"].value_counts(normalize=True).dropna() * 100).round(2)
    display(turno_share.to_frame("% sinistros").T)

    # insuficiência por logradouro
    freq = si["logradouro"].fillna("(sem logradouro)").value_counts()
    print(f"\nLogradouros distintos: {len(freq):,}")
    print(f"  com <5 sinistros em ~11 anos: {(freq < 5).sum():,} ({(freq < 5).sum()/len(freq):.0%})")

    # modelo secundário — mesma grade de 27 combinações
    TOP_LOG = 40
    top_log = freq.head(TOP_LOG).index
    si["logradouro_group"] = si["logradouro"].fillna("(sem logradouro)").str.title()
    si["logradouro_group"] = si["logradouro_group"].where(si["logradouro_group"].isin(top_log), "OUTRAS")
    si["ano"] = si["ano_sinistro"].astype(int)
    si["mes"] = si["mes_sinistro"].astype(int)

    gs = (
        si.dropna(subset=["dia"]).groupby(["logradouro_group", "dia", "faixa", "ano", "mes"])
        .size().rename("ocorrencias").reset_index()
    )
    grade_si = pd.MultiIndex.from_product(
        [sorted(si["logradouro_group"].unique()), DIAS, FAIXAS, sorted(si["ano"].unique()), sorted(si["mes"].unique())],
        names=["logradouro_group", "dia", "faixa", "ano", "mes"],
    ).to_frame(index=False)
    gs = grade_si.merge(gs, on=["logradouro_group", "dia", "faixa", "ano", "mes"], how="left").fillna({"ocorrencias": 0})
    gs["ocorrencias"] = gs["ocorrencias"].astype(int)
    gs["target_log"] = np.log1p(gs["ocorrencias"])

    treino_anos = [y for y in sorted(gs["ano"].unique()) if y <= 2024]
    teste_anos = [y for y in sorted(gs["ano"].unique()) if y > 2024]
    print(f"\n{len(grade_si):,} células → treino {treino_anos[0]}–{treino_anos[-1]} | teste {teste_anos[0]}–{teste_anos[-1]}")

    r_infosiga = treinar_e_avaliar(
        gs, ["logradouro_group", "dia", "faixa"], "target_log", "ano",
        treino_val=treino_anos, teste_val=teste_anos, nome_fonte="Infosiga",
    )

if r_infosiga is not None:
    figuras_base(r_infosiga, "contraponto")

,qtd_pedestre,qtd_bicicleta,qtd_motocicleta,qtd_automovel,qtd_onibus,qtd_caminhao
ano_sinistro,,,,,,
2023,5468,638,19361,20512,2167,1037
2024,5197,626,19655,20908,2275,1023
2025,4705,579,19710,19775,2101,995
2026,3132,540,14754,14577,1485,703


turno,TARDE,NOITE,MANHA,MADRUGADA,NAO DISPONIVEL
% sinistros,31.75,31.46,25.24,11.42,0.13



Logradouros distintos: 33,835
  com <5 sinistros em ~11 anos: 26,358 (78%)



4,368 células → treino 2014–2024 | teste 2025–2026



[Infosiga] melhores hiperparâmetros: {'max_depth': 6, 'min_samples_leaf': 50, 'n_estimators': 200}
[Infosiga] células treino/teste: 3,696 / 672


,RMSE,MAE,R2
treino,1.9961,1.9154,0.0272
teste,1.9702,1.8939,-0.4064


,importância
faixa_MADRUGADA (00-05),0.4416
faixa_NOITE (18-23),0.1431
dia_6,0.0902
faixa_TARDE (12-17),0.0745
faixa_MANHA (06-11),0.0643
dia_5,0.0600


**Leitura** — o modelo secundário usa o mesmo protocolo sobre sinistros por
logradouro: mostra se a estrutura dia × faixa fornece além do crime uma
evidência de **robustez entre naturezas de evento** (lista do relatório final).

## 7. Limitações honestas (obrigatório — subnotificação e lacunas)

| Limitação | Efeito no resultado | Tratamento no produto |
|---|---|---|
| **Subnotificação** — o BO depende de denúncia; furto com perda percebida só às vezes registra | Célula com 0 pode ter ocorrências reais ⇒ **zero ≠ seguro** | Toda estimativa exibe volume de registros que a sustenta; células com dados insuficientes ficam explícitas |
| BOs sem bairro/hora informados | Perda de granularidade (quantificada na §2) | Mostrar taxa de completude como "confiança" |
| SSP-SP só 2026 completo | O split temporal é intra-anual (por mês) — não capta sazonalidade multi-anual | A série longa do Infosiga (2015–2026) compensa na contraprova |
| Infosiga com lat/lon freq. ausentes | Impede análise segmento-a-segmento precisa | Uso por logradouro agregado (já feito) |
| `qtd_*` conta veículos/pessoas envolvidos, não exposição individual | Modo da vítima é inferido com cuidado | Composição de risco descritiva, nunca exposição individual |
| BO pode ser registrado em unidade fora do local do evento | Erro de atribuição territorial | Agregado por bairro, nunca a endereço individual |

## 8. Conclusão — leitura para o cidadão

*(números reais pós-execução: mantidos na §5 — a conclusão usa a comparação,
não veredito territorial)*

- O modelo reproduz a estrutura **bairro × dia × faixa horária** com as métricas
  da §5; a pergunta tem resposta **quantitativa**, comparativa e com medida de
  confiança (volume de dados da célula);
- **Para o cidadão**, o VoltaSegura nunca dirá "o bairro X é perigoso". Ele dirá:
  > *com base em N registros de 2026 deste bairro e faixa, a frequência estimada
  > de roubo/furto está X% acima/abaixo do padrão da cidade nesse horário —
  > com confiança qualificada pelo volume de dados (registros no período).*
- **Próximos passos** (Fase 1): integrar iluminação pública (GeoSampa, já
  validada ao vivo) e posições de ônibus em tempo real (Olho Vivo, já
  autenticada e testada); validar a leitura com a persona (Marina, 22, testes
  de usabilidade — hipótese em aberto no doc vivo).